# ConverterIQ — Exploratory Data Analysis of `id_mea_2`

This notebook documents the initial EDA for the instantaneous three-phase current dataset used by ConverterIQ.

**Primary goals**

1. Validate dataset integrity and label distribution.
2. Understand the temporal acquisition structure.
3. Reconstruct label runs and candidate fault-injection episodes.
4. Verify the waveform layout and physical plausibility.
5. Quantify within-episode redundancy and cross-episode variation.
6. Produce modeling decisions, especially an episode-aware train/validation/test split.

> **EDA scope:** Detailed signal analysis is not repeated for all 47 fault classes at this stage. `id_error=12` is used as a representative case study to test redundancy and cross-episode variation. After a baseline model is trained, confusion matrices and per-class metrics should drive any deeper class-specific EDA.

## 0. Setup

In [1]:
from pathlib import Path
from collections import Counter

import ijson
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = Path("../data/id_mea_2.json")

## 1. Dataset audit

The first pass streams through the JSON file instead of loading the full waveform dataset into memory.  
We verify record count, label counts, waveform length, experiment ID, and device ID.

In [2]:
n_records = 0
label_counts = Counter()
mea_lengths = Counter()
experiment_counts = Counter()
device_counts = Counter()

first_record = None

with DATA_PATH.open("rb") as f:
    for record in ijson.items(f, "item"):
        if first_record is None:
            first_record = record

        n_records += 1
        label_counts[record["id_error"]] += 1
        mea_lengths[len(record["mea_value"])] += 1
        experiment_counts[record.get("experiment_id")] += 1
        device_counts[record.get("device")] += 1

print("Number of records:", n_records)
print("Record keys:", first_record.keys())
print("First id_error:", first_record["id_error"])
print("First id_mea:", first_record["id_mea"])
print("mea_value length distribution:", mea_lengths)

print("\nLabels:")
for label, count in sorted(label_counts.items()):
    print(label, count)

print("\nExperiments:")
print(experiment_counts)

print("\nDevices:")
print(device_counts)

Number of records: 386705
Record keys: dict_keys(['_id', 'timestamp', 'device', 'id_mea', 'id_error', 'mea_value', 'experiment_id', 'store_timestamp'])
First id_error: 0
First id_mea: 2
mea_value length distribution: Counter({60: 386705})

Labels:
0 277041
1 2388
2 2265
3 2306
4 2333
5 2424
6 2375
7 2272
8 2300
9 2202
10 2294
11 2218
12 2445
13 2666
14 2512
15 2578
16 2552
17 2571
18 2631
19 2541
20 2702
21 2654
22 2661
23 2499
24 2494
25 2698
26 2566
27 2588
28 2546
29 2397
30 2277
31 2238
32 2216
33 2312
34 2281
35 2223
36 2326
37 2300
38 2104
39 1989
40 2007
41 1936
42 1959
43 1868
44 2022
45 1967
46 1948
47 2013

Experiments:
Counter({'experiment_3': 386705})

Devices:
Counter({'test1': 386705})


### Dataset audit findings

- **386,705 records** are present.
- Every record contains exactly **60 waveform values**.
- All **48 labels** (`id_error = 0–47`) are present.
- Healthy (`id_error=0`) contains **277,041 records (~71.6%)**; all faults together contain **109,664 records (~28.4%)**.
- Individual fault classes are relatively balanced compared with one another (roughly 1.9k–2.7k records per class).
- All records come from `experiment_3` and device `test1`, so these fields cannot be used as natural grouping variables for train/test splitting.

**Modeling implication:** the primary class imbalance is healthy versus each individual fault class. Overall accuracy alone will therefore be insufficient; macro-F1 and per-class recall should be emphasized later.

## 2. Temporal acquisition structure

Only metadata are loaded into a pandas DataFrame. This keeps waveform data on disk while allowing timestamp, label-transition, and episode analysis.

In [3]:
rows = []

with DATA_PATH.open("rb") as f:
    for i, record in enumerate(ijson.items(f, "item")):
        rows.append({
            "row_id": i,
            "timestamp": record["timestamp"],
            "store_timestamp": record["store_timestamp"],
            "id_error": record["id_error"],
        })

meta = pd.DataFrame(rows)
meta.info()

<class 'pandas.DataFrame'>
RangeIndex: 386705 entries, 0 to 386704
Data columns (total 4 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   row_id           386705 non-null  int64 
 1   timestamp        386705 non-null  object
 2   store_timestamp  386705 non-null  object
 3   id_error         386705 non-null  int64 
dtypes: int64(2), object(2)
memory usage: 11.8+ MB


`ijson` parses JSON numbers as `Decimal`, so timestamps are first converted to floating-point Unix seconds and then to UTC datetimes.  
The original timestamp columns are preserved.

In [4]:
meta["timestamp_num"] = meta["timestamp"].astype(float)
meta["store_timestamp_num"] = meta["store_timestamp"].astype(float)

meta["timestamp_dt"] = pd.to_datetime(
    meta["timestamp_num"],
    unit="s",
    utc=True
)

meta["store_timestamp_dt"] = pd.to_datetime(
    meta["store_timestamp_num"],
    unit="s",
    utc=True
)

### 2.1 Normal inter-record cadence

In [5]:
meta["dt_seconds"] = meta["timestamp_num"].diff()

print("Timestamp monotonic:")
print(meta["timestamp_num"].is_monotonic_increasing)

print("\nDuplicate timestamps:")
print(meta["timestamp_num"].duplicated().sum())

print("\nTime-gap summary:")
print(meta["dt_seconds"].describe())

print("\nMost common time gaps:")
print(meta["dt_seconds"].round(3).value_counts().head(20))

Timestamp monotonic:
True

Duplicate timestamps:
0

Time-gap summary:
count    386704.000000
mean          1.028919
std          17.923744
min           0.071163
25%           0.998573
50%           0.999992
75%           1.001433
max       11146.931422
Name: dt_seconds, dtype: float64

Most common time gaps:
dt_seconds
1.000    75217
0.999    63318
1.001    62155
0.998    41927
1.002    41507
1.003    25192
0.997    24722
1.004    13817
0.996    13625
0.995     6562
1.005     6545
1.006     3028
0.994     2957
0.993     1458
1.007     1423
0.992      700
1.008      698
1.009      388
0.991      367
1.010      220
Name: count, dtype: int64


In [6]:
meta["dt_seconds"].quantile([
    0.99,
    0.995,
    0.999,
    0.9995,
    0.9999
])

0.9900    1.006140
0.9950    1.007232
0.9990    1.010139
0.9995    1.011779
0.9999    1.019106
Name: dt_seconds, dtype: float64

The median timestamp gap is essentially **1 second**, and even the 99.99th percentile is only about **1.019 s**.  
Thus, the normal logging cadence is approximately **1 Hz**.

### 2.2 Measurement timestamp vs. storage timestamp

In [7]:
meta["store_delay"] = (
    meta["store_timestamp_num"]
    - meta["timestamp_num"]
)

print(meta["store_delay"].describe())

count    386705.000000
mean          0.004899
std           0.222752
min           0.002457
25%           0.002784
50%           0.002855
75%           0.002953
max          37.556698
Name: store_delay, dtype: float64


The median storage delay is only a few milliseconds, but rare storage-delay outliers exist.  
For sequence analysis, the measurement `timestamp` is therefore used rather than `store_timestamp`.

### 2.3 Rare acquisition gaps

In [8]:
meta["prev_id_error"] = meta["id_error"].shift()

thresholds = [1.05, 1.1, 1.2, 1.5, 2, 5, 10, 60]

for t in thresholds:
    mask = meta["dt_seconds"] > t

    total = mask.sum()
    same_label = (
        mask &
        (meta["id_error"] == meta["prev_id_error"])
    ).sum()

    changed_label = (
        mask &
        (meta["id_error"] != meta["prev_id_error"])
    ).sum()

    print(
        f"> {t:>5} sec: "
        f"{total:>6} gaps | "
        f"same label: {same_label:>6} | "
        f"label changed: {changed_label:>6}"
    )

>  1.05 sec:     21 gaps | same label:     20 | label changed:      1
>   1.1 sec:     17 gaps | same label:     17 | label changed:      0
>   1.2 sec:     15 gaps | same label:     15 | label changed:      0
>   1.5 sec:     15 gaps | same label:     15 | label changed:      0
>     2 sec:     12 gaps | same label:     12 | label changed:      0
>     5 sec:      2 gaps | same label:      2 | label changed:      0
>    10 sec:      2 gaps | same label:      2 | label changed:      0
>    60 sec:      1 gaps | same label:      1 | label changed:      0


In [9]:
GAP_THRESHOLD = 5
centers = meta.index[meta["dt_seconds"] > GAP_THRESHOLD]
print(centers.tolist())

for center in centers:
    display(
        meta.loc[
            center-5:center+5,
            [
                "row_id",
                "timestamp_dt",
                "id_error",
                "dt_seconds",
            ]
        ]
    )

[137990, 214427]


,row_id,timestamp_dt,id_error,dt_seconds
137985,137985,2026-05-14 11:54:04.947927475+00:00,0,1.001076
137986,137986,2026-05-14 11:54:05.945552826+00:00,0,0.997625
137987,137987,2026-05-14 11:54:06.948991299+00:00,0,1.003438
137988,137988,2026-05-14 11:54:07.943994999+00:00,0,0.995004
137989,137989,2026-05-14 11:54:08.945451260+00:00,0,1.001456
137990,137990,2026-05-14 14:59:55.876873016+00:00,0,11146.931422
137991,137991,2026-05-14 14:59:56.875179291+00:00,0,0.998306
137992,137992,2026-05-14 14:59:57.878807783+00:00,0,1.003628
137993,137993,2026-05-14 14:59:58.876052380+00:00,0,0.997245
137994,137994,2026-05-14 14:59:59.874919653+00:00,0,0.998867


,row_id,timestamp_dt,id_error,dt_seconds
214422,214422,2026-05-15 12:13:47.915809393+00:00,0,1.000772
214423,214423,2026-05-15 12:13:48.917181969+00:00,0,1.001373
214424,214424,2026-05-15 12:13:49.913019657+00:00,0,0.995838
214425,214425,2026-05-15 12:13:50.916508198+00:00,0,1.003489
214426,214426,2026-05-15 12:13:51.913485289+00:00,0,0.996977
214427,214427,2026-05-15 12:14:22.917989016+00:00,0,31.004504
214428,214428,2026-05-15 12:14:23.912441492+00:00,0,0.994452
214429,214429,2026-05-15 12:14:24.914433479+00:00,0,1.001992
214430,214430,2026-05-15 12:14:25.915557623+00:00,0,1.001124
214431,214431,2026-05-15 12:14:26.914866447+00:00,0,0.999309


### Temporal sampling conclusion

Records are chronologically ordered with no duplicate timestamps. Under normal acquisition, records are logged at approximately **1 Hz**; 99.99% of consecutive gaps are below approximately **1.019 s**.

A small number of acquisition interruptions exist, including pauses of roughly **31 s** and **3.1 h**. Inspection around the two largest gaps shows that the label remains healthy before and after the interruption and that the normal ~1 s cadence immediately resumes. These gaps are therefore more consistent with logging/acquisition pauses than fault transitions.

Each row should be interpreted as a **one-cycle waveform snapshot sampled approximately once per second**, rather than as part of one continuous 1 kHz waveform stream.

## 3. Label runs and candidate fault episodes

Three levels are distinguished:

- **Raw record:** one row in the JSON file.
- **Label run:** a contiguous block of rows with the same `id_error`.
- **Fault episode:** a non-zero label run, interpreted as one candidate fault injection.

For example, a raw sequence such as `0,0,0,12,12,12,0,0` becomes the label-run sequence `0 → 12 → 0`.

In [ ]:
meta["label_changed"] = (
    meta["id_error"] != meta["id_error"].shift()
)

transitions = meta.loc[
    meta["label_changed"],
    ["row_id", "timestamp_dt", "id_error", "dt_seconds"]
]

print("Number of label runs:", len(transitions))
print(transitions.head(30))

In [ ]:
meta["new_label_run"] = (
    meta["id_error"] != meta["id_error"].shift()
)

meta["label_run_id"] = meta["new_label_run"].cumsum()

In [ ]:
run_summary = (
    meta.groupby("label_run_id")
    .agg(
        id_error=("id_error", "first"),
        n_records=("id_error", "size"),
        start_row=("row_id", "first"),
        end_row=("row_id", "last"),
        start_time=("timestamp_dt", "first"),
        end_time=("timestamp_dt", "last"),
    )
    .reset_index()
)

run_summary["duration_seconds"] = (
    run_summary["end_time"]
    - run_summary["start_time"]
).dt.total_seconds()

In [ ]:
print("Number of label runs:", len(run_summary))

print("\nRun summary:")
print(run_summary["n_records"].describe())

### 3.1 Healthy versus fault run lengths

In [ ]:
healthy_runs = run_summary[
    run_summary["id_error"] == 0
]

fault_runs = run_summary[
    run_summary["id_error"] != 0
]

print("Healthy runs:", len(healthy_runs))
print("Fault runs:", len(fault_runs))

print("\nHealthy run lengths:")
print(healthy_runs["n_records"].describe())

print("\nFault run lengths:")
print(fault_runs["n_records"].describe())

### 3.2 Episode counts by label

In [ ]:
per_class_runs = (
    run_summary
    .groupby("id_error")
    .agg(
        n_runs=("label_run_id", "count"),
        total_records=("n_records", "sum"),
        mean_records_per_run=("n_records", "mean"),
        median_records_per_run=("n_records", "median"),
        min_records_per_run=("n_records", "min"),
        max_records_per_run=("n_records", "max"),
    )
)

per_class_runs

### 3.3 Transition structure

In [ ]:
run_summary["prev_label"] = run_summary["id_error"].shift()
run_summary["next_label"] = run_summary["id_error"].shift(-1)

In [ ]:
healthy_to_fault = (
    (run_summary["prev_label"] == 0)
    & (run_summary["id_error"] != 0)
).sum()

fault_to_healthy = (
    (run_summary["prev_label"] != 0)
    & (run_summary["id_error"] == 0)
).sum()

fault_to_fault = (
    (run_summary["prev_label"] != 0)
    & (run_summary["id_error"] != 0)
).sum()

print("Healthy -> Fault:", healthy_to_fault)
print("Fault -> Healthy:", fault_to_healthy)
print("Fault -> Fault:", fault_to_fault)

In [ ]:
fault_run_context = run_summary[
    run_summary["id_error"] != 0
].copy()

fault_run_context["healthy_before"] = (
    fault_run_context["prev_label"] == 0
)

fault_run_context["healthy_after"] = (
    fault_run_context["next_label"] == 0
)

print(
    fault_run_context[
        ["healthy_before", "healthy_after"]
    ].value_counts()
)

The run sequence alternates cleanly between healthy and fault states:

- **Healthy → Fault:** 16,813
- **Fault → Healthy:** 16,814
- **Fault → Fault:** 0

Every fault run is preceded and followed by a healthy run. This strongly supports treating each contiguous non-zero run as one **candidate fault-injection episode**.

In [ ]:
fault_episodes = (
    run_summary[run_summary["id_error"] != 0]
    .copy()
    .reset_index(drop=True)
)

fault_episodes.insert(
    0,
    "fault_episode_id",
    np.arange(1, len(fault_episodes) + 1)
)

print("Fault episode table shape:", fault_episodes.shape)
display(fault_episodes.head())

print("\nFault episodes per class:")
display(fault_episodes["id_error"].value_counts().sort_index())

### Episode-structure conclusion

The dataset contains **16,813 candidate fault episodes**. Most fault episodes contain approximately **6–7 recorded snapshots** (minimum 2, maximum 11), while healthy runs are longer (median approximately 16 records).

Most fault classes contain roughly **300–400 separate fault episodes**, so their record counts are not produced by only a few long events.

**Critical modeling implication:** train/validation/test splitting must operate at the **episode level**, not the row level.

## 4. Waveform layout validation

Every `mea_value` contains 60 numbers. Two possible layouts were tested:

1. **Block-wise:** 20 Phase-A samples, then 20 Phase-B samples, then 20 Phase-C samples.
2. **Interleaved:** A0, B0, C0, A1, B1, C1, ...

A healthy record is used to visually determine which reconstruction produces a physically plausible balanced three-phase waveform.

In [ ]:
with DATA_PATH.open("rb") as f:
    first_record = next(ijson.items(f, "item"))

values = np.asarray(first_record["mea_value"], dtype=float)

wave_block = values.reshape(3, 20).T
wave_interleaved = values.reshape(20, 3)

print("Label:", first_record["id_error"])
print("Raw shape:", values.shape)
print("Candidate waveform shape:", wave_block.shape)

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(wave_block[:, 0], marker="o", label="Phase A")
plt.plot(wave_block[:, 1], marker="o", label="Phase B")
plt.plot(wave_block[:, 2], marker="o", label="Phase C")

plt.xlabel("Sample")
plt.ylabel("Current")
plt.title("Candidate 1: Block-wise reshape")
plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(wave_interleaved[:, 0], marker="o", label="Phase A")
plt.plot(wave_interleaved[:, 1], marker="o", label="Phase B")
plt.plot(wave_interleaved[:, 2], marker="o", label="Phase C")

plt.xlabel("Sample")
plt.ylabel("Current")
plt.title("Candidate 2: Interleaved reshape")
plt.legend()
plt.grid(alpha=0.3)

plt.show()

The **block-wise reconstruction** produces the expected smooth three-phase sinusoidal structure, whereas the interleaved interpretation produces discontinuous traces.  
Therefore, the waveform is stored as:

`A0 ... A19, B0 ... B19, C0 ... C19`.

In [ ]:
def reshape_waveform(mea_value):
    values = np.asarray(mea_value, dtype=float)

    if len(values) != 60:
        raise ValueError(
            f"Expected 60 values, got {len(values)}"
        )

    return values.reshape(3, 20).T

## 5. Representative case study: `id_error=12`

A single fault class is examined in detail to answer two modeling questions:

1. Are multiple rows from the **same fault episode** near-duplicates?
2. Do **independent episodes** of the same fault still exhibit variation?

This case study is not intended to characterize all 47 fault classes.

### 5.1 Healthy waveform versus one fault episode

In [ ]:
target_rows = set(range(6, 14))

selected_records = {}

with DATA_PATH.open("rb") as f:
    for i, record in enumerate(ijson.items(f, "item")):

        if i in target_rows:
            selected_records[i] = record

        if i > max(target_rows):
            break


for row_id, record in selected_records.items():
    print(
        row_id,
        "label =", record["id_error"]
    )

In [ ]:
healthy_wave = reshape_waveform(
    selected_records[6]["mea_value"]
)

fault_wave = reshape_waveform(
    selected_records[7]["mea_value"]
)



plt.figure(figsize=(10, 5))

for phase in range(3):
    plt.plot(
        healthy_wave[:, phase],
        marker="o",
        label=f"Phase {chr(65 + phase)}"
    )

plt.title("Healthy waveform immediately before fault")
plt.xlabel("Sample")
plt.ylabel("Current")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

for phase in range(3):
    plt.plot(
        fault_wave[:, phase],
        marker="o",
        label=f"Phase {chr(65 + phase)}"
    )

plt.title("Fault waveform: id_error = 12")
plt.xlabel("Sample")
plt.ylabel("Current")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

The first examined `id_error=12` episode shows a large change in current magnitude while remaining approximately sinusoidal and zero-centered.  
This is only a case-study observation; it should not be generalized to every `id_error=12` episode without considering operating-condition variation.

In [ ]:
def waveform_features(wave):
    return {
        "rms": np.sqrt(np.mean(wave**2, axis=0)),
        "mean": np.mean(wave, axis=0),
        "peak_abs": np.max(np.abs(wave), axis=0),
        "peak_to_peak": np.ptp(wave, axis=0),
    }


healthy_features = waveform_features(healthy_wave)
fault_features = waveform_features(fault_wave)

for name in healthy_features:
    print(f"\n{name}")
    print("Healthy:", healthy_features[name])
    print("Fault 12:", fault_features[name])

In [ ]:
episode_features = []

for row_id in range(7, 14):
    wave = reshape_waveform(
        selected_records[row_id]["mea_value"]
    )

    rms = np.sqrt(np.mean(wave**2, axis=0))

    episode_features.append({
        "row_id": row_id,
        "id_error": selected_records[row_id]["id_error"],
        "rms_A": rms[0],
        "rms_B": rms[1],
        "rms_C": rms[2],
    })

episode_features = pd.DataFrame(episode_features)

episode_features

### 5.2 Within-episode redundancy

In [ ]:
plt.figure(figsize=(10, 5))

for row_id in range(7, 14):
    wave = reshape_waveform(
        selected_records[row_id]["mea_value"]
    )

    plt.plot(
        wave[:, 0],
        alpha=0.7,
        label=f"row {row_id}"
    )

plt.xlabel("Sample")
plt.ylabel("Current")
plt.title("Fault episode 1 — Phase A, id_error = 12")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
episode_phase_a = []

for row_id in range(7, 14):
    wave = reshape_waveform(
        selected_records[row_id]["mea_value"]
    )
    episode_phase_a.append(wave[:, 0])

episode_phase_a = np.array(episode_phase_a)

corr_matrix = np.corrcoef(episode_phase_a)

print(np.round(corr_matrix, 6))

In [ ]:
upper_triangle = corr_matrix[
    np.triu_indices_from(corr_matrix, k=1)
]

print("Mean pairwise correlation:",
      upper_triangle.mean())

print("Min pairwise correlation:",
      upper_triangle.min())

print("Max pairwise correlation:",
      upper_triangle.max())

In [ ]:
reference = episode_phase_a[0]

reference_rms = np.sqrt(
    np.mean(reference**2)
)

for i, row_id in enumerate(range(7, 14)):
    rmse = np.sqrt(
        np.mean(
            (episode_phase_a[i] - reference) ** 2
        )
    )

    nrmse = rmse / reference_rms

    print(
        f"row {row_id}: "
        f"NRMSE = {nrmse:.6f}"
    )


### Within-episode finding

For the examined seven-record fault episode:

- Mean pairwise Phase-A correlation is approximately **0.999985**.
- Minimum pairwise correlation remains above **0.99995**.
- NRMSE relative to the first record remains below approximately **1.5%**.

The records are therefore near-duplicates at the waveform level. This provides direct evidence that a random row-level split would leak highly correlated observations across training and evaluation sets.

## 6. Cross-episode variation for the same fault class

To avoid mixing within-episode duplication with genuine repeated experiments, one representative waveform (the first row) is selected from each of the **370 independent `id_error=12` episodes**.

In [ ]:
fault12_eps = fault_episodes[
    fault_episodes["id_error"] == 12
].copy()

print(len(fault12_eps))
fault12_eps.head()


In [ ]:
fault12_start_rows = set(
    fault12_eps["start_row"].astype(int)
)

fault12_representatives = {}

with DATA_PATH.open("rb") as f:
    for i, record in enumerate(ijson.items(f, "item")):

        if i in fault12_start_rows:
            fault12_representatives[i] = record

        if len(fault12_representatives) == len(fault12_start_rows):
            break

print(len(fault12_representatives))


In [ ]:
fault12_phase_a = []

for row_id in sorted(fault12_representatives):
    wave = reshape_waveform(
        fault12_representatives[row_id]["mea_value"]
    )

    fault12_phase_a.append(wave[:, 0])

fault12_phase_a = np.array(fault12_phase_a)

print(fault12_phase_a.shape)

### 6.1 Visual variation across independent episodes

In [ ]:
rng = np.random.default_rng(42)

idx = rng.choice(
    len(fault12_phase_a),
    size=20,
    replace=False
)

plt.figure(figsize=(10, 5))

for i in idx:
    plt.plot(
        fault12_phase_a[i],
        alpha=0.4
    )

plt.title(
    "Fault 12 — Phase A across 20 independent episodes"
)
plt.xlabel("Sample")
plt.ylabel("Current")
plt.grid(alpha=0.3)
plt.show()

Independent episodes preserve similar periodic morphology but begin at different electrical phases.  
Raw pointwise correlation is therefore not an appropriate cross-episode similarity measure without phase alignment.

In [ ]:
def best_circular_alignment(reference, wave):
    best_corr = -np.inf
    best_shift = None
    best_wave = None

    for shift in range(len(wave)):
        shifted = np.roll(wave, shift)

        corr = np.corrcoef(
            reference,
            shifted
        )[0, 1]

        if corr > best_corr:
            best_corr = corr
            best_shift = shift
            best_wave = shifted

    return best_corr, best_shift, best_wave

In [ ]:
reference = fault12_phase_a[0]

aligned_corrs = []
aligned_nrmses = []
best_shifts = []

reference_rms = np.sqrt(
    np.mean(reference**2)
)

for wave in fault12_phase_a[1:]:

    corr, shift, aligned_wave = (
        best_circular_alignment(
            reference,
            wave
        )
    )

    rmse = np.sqrt(
        np.mean(
            (aligned_wave - reference) ** 2
        )
    )

    aligned_corrs.append(corr)
    aligned_nrmses.append(
        rmse / reference_rms
    )
    best_shifts.append(shift)


print("Phase-aligned correlation:")
print(pd.Series(aligned_corrs).describe())

print("\nPhase-aligned NRMSE:")
print(pd.Series(aligned_nrmses).describe())

print("\nBest circular shifts:")
print(pd.Series(best_shifts).value_counts().sort_index())

pd.Series(best_shifts).value_counts().sort_index()

After integer-sample circular alignment, the Phase-A shape remains highly similar across independent episodes:

- Mean aligned correlation ≈ **0.9956**
- Median aligned correlation ≈ **0.9952**
- Minimum aligned correlation ≈ **0.9877**

The best shifts cluster at a few discrete sample positions, indicating structured variation in acquisition phase.

### 6.2 RMS variation across independent episodes

In [ ]:
fault12_rms_a = np.sqrt(
    np.mean(fault12_phase_a**2, axis=1)
)

print(
    pd.Series(fault12_rms_a).describe()
)

In [ ]:
plt.figure(figsize=(9, 5))

plt.hist(
    fault12_rms_a,
    bins=30
)

plt.xlabel("Phase A RMS Current")
plt.ylabel("Number of Episodes")
plt.title("Fault 12 — Phase A RMS Across Independent Episodes")
plt.grid(alpha=0.3)

plt.show()

The RMS distribution is clearly structured rather than a single narrow peak. This shows that independent episodes of the same fault class can occur at different current magnitudes.

This is enough for the initial EDA: the model should not be assumed to operate under one fixed amplitude regime, and deeper investigation of individual fault classes should be deferred until baseline-model errors identify where it is useful.

# 7. Data-quality and outlier audit

The purpose of this section is **not** to remove statistically extreme waveforms automatically.

In a fault-diagnosis dataset, large amplitude, imbalance, and distortion may be legitimate signal signatures. We therefore separate:

- **invalid numerical records**: NaN, Inf, or obvious rail/saturation artifacts;
- **statistically extreme but finite records**: retained unless there is direct evidence of corruption.

Generic z-score or IQR filtering is intentionally avoided.


In [ ]:
def waveform_stats(mea_value):
    wave = np.asarray(mea_value, dtype=float).reshape(3, 20).T

    return {
        "rms_max": np.sqrt(np.mean(wave**2, axis=0)).max(),
        "peak_abs": np.abs(wave).max(),
        "wave_std": wave.std(),
    }

In [ ]:
stats = []

with DATA_PATH.open("rb") as f:
    for i, record in enumerate(ijson.items(f, "item")):

        values = np.asarray(record["mea_value"], dtype=float)

        stats.append({
            "row_id": i,
            "id_error": int(record["id_error"]),
            "has_nan": np.isnan(values).any(),
            "has_inf": np.isinf(values).any(),
            **waveform_stats(values),
        })

outlier_df = pd.DataFrame(stats)

In [ ]:
print("NaN rows:", outlier_df["has_nan"].sum())
print("Inf rows:", outlier_df["has_inf"].sum())

print(outlier_df[
    ["rms_max", "peak_abs", "wave_std"]
].describe(percentiles=[
    0.01, 0.05, 0.95, 0.99, 0.999
]))

In [ ]:
healthy_stats = outlier_df[
    outlier_df["id_error"] == 0
]

healthy_stats["rms_max"].describe(
    percentiles=[0.001, 0.01, 0.99, 0.999]
)

In [ ]:
inspect_rows = {83865, 83192, 137980}

inspect_records = {}

with DATA_PATH.open("rb") as f:
    for i, record in enumerate(ijson.items(f, "item")):

        if i in inspect_rows:
            inspect_records[i] = record

        if len(inspect_records) == len(inspect_rows):
            break

In [ ]:
for row_id, record in inspect_records.items():

    wave = reshape_waveform(record["mea_value"])

    plt.figure(figsize=(9, 4))

    plt.plot(wave[:, 0], marker="o", label="Phase A")
    plt.plot(wave[:, 1], marker="o", label="Phase B")
    plt.plot(wave[:, 2], marker="o", label="Phase C")

    plt.title(
        f"Extreme waveform — row {row_id}, "
        f"id_error={record['id_error']}"
    )
    plt.xlabel("Sample")
    plt.ylabel("Current")
    plt.legend()
    plt.grid(alpha=0.3)

    plt.show()

### 7.1 Numerical saturation flag

Inspection of the most extreme waveforms shows repeated clipping near approximately ±\(2^{30}\), including both healthy and fault-labeled records. This pattern is qualitatively different from an ordinary high-amplitude sinusoid and is treated as a numerical-quality flag.

The detector below flags records containing at least one sample within 1% of ±\(2^{30}\). The original JSON is **not modified**; the flag is carried forward so downstream modeling can exclude these rows reproducibly.


In [ ]:
SAT_VALUE = 2**30
SAT_TOL = 0.01 * SAT_VALUE   # 1% tolerance

def is_saturated(mea_value):
    values = np.asarray(mea_value, dtype=float)

    return np.any(
        np.abs(np.abs(values) - SAT_VALUE) <= SAT_TOL
    )

In [ ]:
quality_rows = []

with DATA_PATH.open("rb") as f:
    for i, record in enumerate(ijson.items(f, "item")):
        values = np.asarray(record["mea_value"], dtype=float)

        quality_rows.append({
            "row_id": i,
            "id_error": int(record["id_error"]),
            "is_saturated": is_saturated(values),
        })

quality_df = pd.DataFrame(quality_rows)

In [ ]:
print(
    quality_df["is_saturated"].value_counts()
)

print(
    quality_df["is_saturated"]
    .value_counts(normalize=True)
)

print("\nBy label:")
print(
    quality_df.loc[
        quality_df["is_saturated"],
        "id_error"
    ]
    .value_counts()
    .sort_index()
)

# 8. EDA summary and modeling implications

## Data integrity

- `id_mea_2` contains **386,705 records**.
- Every record has exactly **60 values = 20 samples × 3 phases**.
- All **48 labels** are represented.
- Healthy (`id_error=0`) accounts for approximately **71.6%** of records; individual fault classes are comparatively balanced.
- All rows belong to one experiment/device combination, so `experiment_id` and `device` do not provide natural train/test groups.
- No NaN or infinite waveform values were observed.

## Temporal and episode structure

- Timestamps are strictly increasing with no duplicates.
- Normal logging cadence is approximately **1 Hz**; each row itself contains one complete 20 ms electrical-cycle waveform.
- The data contain **33,627 label runs**: 16,814 healthy runs and 16,813 fault runs.
- The run sequence alternates strictly between healthy and fault states; no Fault→Fault run transition occurs.
- A contiguous non-zero label run is therefore a reasonable **candidate fault-injection episode**.
- Fault classes generally contain **300–400 independent episodes**.

## Waveform structure

- The correct raw layout is block-wise:
  `A0...A19, B0...B19, C0...C19`.
- Healthy records show a physically plausible three-phase sinusoidal pattern.

## Leakage evidence

A representative `id_error=12` episode shows extremely strong within-episode redundancy:

- Mean pairwise Phase-A correlation ≈ **0.999985**
- Minimum correlation > **0.99995**
- NRMSE relative to the first record < **1.5%**

Therefore, a random row-level train/test split would place near-duplicate waveform snapshots on both sides of the split and produce overly optimistic evaluation.

## Cross-episode variation

Independent episodes of the same fault class are not exact copies. They vary in starting phase and current magnitude while preserving strongly similar periodic morphology after circular alignment. This supports a model that learns waveform structure rather than a fixed starting phase.

## Data-quality / corruption audit

- No NaN or Inf records were found.
- **604 records (~0.156%)** exhibit clipping/saturation-like values near ±\(2^{30}\).
- These records are flagged for downstream exclusion.
- Other statistically extreme but finite waveforms are retained because amplitude and distortion may carry legitimate operating/fault information.

## Modeling consequences

| EDA finding | Modeling consequence |
|---|---|
| Healthy class dominates record count | Report macro-F1 and per-class recall in addition to accuracy |
| Same-episode rows are near-duplicates | **Split by episode/block, never randomly by row** |
| Independent episodes vary in phase | Do not assume a fixed waveform starting phase |
| Absolute amplitude varies across conditions | Do not classify using RMS alone; preserve amplitude information |
| Input is a structured 20×3 time series | 1D CNN / TCN is a reasonable modeling direction |
| 604 saturation-like rows are flagged | Carry a reproducible quality flag into the modeling dataset |

# 9. Preprocessing status: what is done vs. deferred

### Completed in EDA

- Schema / waveform-length validation
- Label-count and class-balance audit
- Timestamp ordering and acquisition-gap audit
- Label-run / candidate episode reconstruction
- Waveform storage-order validation
- Within-episode leakage analysis
- Cross-episode phase/amplitude variation check
- NaN / Inf audit
- Numerical-saturation quality flag

### Intentionally deferred until **after the train/validation/test split**

These steps must not be estimated from the full dataset:

1. **Scaling / standardization**  
   Compute channel-wise scaling statistics from the **training split only**, then apply the same transform to validation and test data. Do not normalize every waveform independently because absolute amplitude contains diagnostic information.

2. **Class weighting or resampling**  
   Decide from the training set only. Do not rebalance validation/test sets.

3. **Phase-shift augmentation**  
   If used, apply only to training data. This is a model-robustness choice, not an EDA cleaning step.

4. **Conversion to modeling format**  
   Convert valid waveforms to `float32` arrays with shape `(N, 20, 3)` after the fixed split is assigned.

5. **Optional label restructuring / fault-family modeling**  
   Keep the original 48-class target for the first baseline. Only introduce grouped or hierarchical labels if baseline confusion patterns justify it.

### Not recommended at this stage

- Generic z-score/IQR outlier removal
- Per-record RMS normalization
- Random row-level splitting
- Automatic removal of first/last records in an episode
- FFT-based feature engineering before establishing a raw-waveform baseline

### Next step

Create the fixed **episode/block-aware train / validation / test split**, carry the data-quality flag into the split manifest, and then build the baseline modeling arrays.
